# Face Swap (Colab)

얼굴만 따서 다른 사진에 합성하는 노트북. 각 셀을 위에서 아래로 순서대로 실행하세요.

**한 번만 실행하면 되는 것 (셀 1~3)**
1. 라이브러리 설치
2. 스왑 모델 다운로드 (~554MB)
3. 얼굴 검출/스왑 파이프라인 정의

**매번 실행 (셀 4~6)**
4. 소스 사진 업로드 (얼굴을 **가져올** 사람)
5. 타깃 사진 업로드 (얼굴을 **바꿀** 사진 — 몸/배경 등)
6. 스왑 실행 + 결과 다운로드

> **사용 시 주의**: 등장하는 사람의 동의를 받은 사진에만 사용하세요. 실제 인물을 사칭하거나, 성적 콘텐츠 제작, 기만적 자료 제작에 사용하지 마세요. 한국을 포함한 여러 국가에서 비동의 합성물은 형사처벌 대상입니다.


## 1. 라이브러리 설치


In [ ]:
!pip install -q insightface==0.7.3 onnxruntime==1.16.3 opencv-python-headless==4.10.0.84 numpy


## 2. 스왑 모델 다운로드


In [ ]:
import hashlib, urllib.request, os
from pathlib import Path

MODEL_URLS = [
    'https://huggingface.co/deepinsight/inswapper/resolve/main/inswapper_128.onnx',
    'https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx',
]
MODEL_SHA256 = 'e4a3f08c753cb72d04e10aa0f7dbe3deebbf39567d4ead6dce08e98aa49e16af'
MODEL_PATH = Path('/content/models/inswapper_128.onnx')
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

if MODEL_PATH.exists() and sha256(MODEL_PATH) == MODEL_SHA256:
    print('이미 받아져 있음:', MODEL_PATH)
else:
    for url in MODEL_URLS:
        try:
            print('다운로드:', url)
            urllib.request.urlretrieve(url, MODEL_PATH)
            break
        except Exception as e:
            print('  실패:', e)
    got = sha256(MODEL_PATH)
    assert got == MODEL_SHA256, f'체크섬 불일치: {got}'
    print('저장:', MODEL_PATH)


## 3. 얼굴 검출/스왑 파이프라인 정의


In [ ]:
from dataclasses import dataclass
from typing import List, Optional
import cv2, numpy as np, insightface

@dataclass
class DetectedFace:
    bbox: np.ndarray
    kps: np.ndarray
    det_score: float
    embedding: np.ndarray
    raw: object
    @property
    def area(self):
        x1,y1,x2,y2 = self.bbox
        return float(max(0.0, x2-x1) * max(0.0, y2-y1))

class FaceDetector:
    def __init__(self, det_size=640):
        self.app = insightface.app.FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'])
        self.app.prepare(ctx_id=0, det_size=(det_size, det_size))
    def detect(self, img_bgr):
        return [DetectedFace(
            bbox=np.asarray(f.bbox, np.float32),
            kps=np.asarray(f.kps, np.float32),
            det_score=float(getattr(f, 'det_score', 0.0)),
            embedding=np.asarray(getattr(f, 'normed_embedding', getattr(f, 'embedding', None)), np.float32),
            raw=f,
        ) for f in self.app.get(img_bgr)]
    def largest(self, faces):
        return max(faces, key=lambda f: f.area) if faces else None

class FaceSwapper:
    def __init__(self, model_path):
        self.swapper = insightface.model_zoo.get_model(str(model_path), providers=['CPUExecutionProvider'])
    def swap(self, img_bgr, target_face, source_face):
        return self.swapper.get(img_bgr, target_face.raw, source_face.raw, paste_back=True)

print('모델 로딩 중... (처음 실행 시 buffalo_l 검출기도 자동 다운로드됨)')
detector = FaceDetector()
swapper = FaceSwapper(MODEL_PATH)
print('준비 완료')


## 4. 소스 사진 업로드
**얼굴을 가져올** 사진 (얼굴 정면이 잘 보이는 사진 1장).


In [ ]:
from google.colab import files
print('소스 사진 (얼굴 가져올 사진) 업로드')
src_uploaded = files.upload()
SRC_PATH = '/content/source.' + list(src_uploaded.keys())[0].rsplit('.', 1)[-1]
with open(SRC_PATH, 'wb') as f:
    f.write(next(iter(src_uploaded.values())))
print('저장:', SRC_PATH)


## 5. 타깃 사진 업로드
**얼굴을 바꿀** 사진 (몸/포즈/배경이 있는 사진). 여러 얼굴이 있으면 기본으로 제일 큰 얼굴 하나만 바꿉니다.


In [ ]:
print('타깃 사진 (얼굴 바꿀 사진) 업로드')
tgt_uploaded = files.upload()
TGT_PATH = '/content/target.' + list(tgt_uploaded.keys())[0].rsplit('.', 1)[-1]
with open(TGT_PATH, 'wb') as f:
    f.write(next(iter(tgt_uploaded.values())))
print('저장:', TGT_PATH)


## 6. 스왑 실행 + 결과 다운로드
결과 이미지가 미리보기로 뜨고, 브라우저에서 `output.jpg`로 자동 다운로드됩니다.

타깃 사진의 **모든 얼굴**을 바꾸고 싶으면 아래 `REPLACE_ALL_TARGET_FACES = True`로 바꾸세요.


In [ ]:
REPLACE_ALL_TARGET_FACES = False

src_img = cv2.imread(SRC_PATH, cv2.IMREAD_COLOR)
tgt_img = cv2.imread(TGT_PATH, cv2.IMREAD_COLOR)
assert src_img is not None, '소스 사진을 읽을 수 없어요'
assert tgt_img is not None, '타깃 사진을 읽을 수 없어요'

src_faces = detector.detect(src_img)
tgt_faces = detector.detect(tgt_img)
print(f'소스에서 얼굴 {len(src_faces)}개, 타깃에서 얼굴 {len(tgt_faces)}개 검출')
assert src_faces, '소스 사진에서 얼굴을 못 찾았어요. 더 정면·고해상도 사진으로 시도해보세요.'
assert tgt_faces, '타깃 사진에서 얼굴을 못 찾았어요.'

src_face = detector.largest(src_faces)
to_replace = tgt_faces if REPLACE_ALL_TARGET_FACES else [detector.largest(tgt_faces)]

result = tgt_img.copy()
for tf in to_replace:
    result = swapper.swap(result, tf, src_face)

OUT_PATH = '/content/output.jpg'
cv2.imwrite(OUT_PATH, result)
print('결과 저장:', OUT_PATH)

from IPython.display import Image, display
display(Image(OUT_PATH))

files.download(OUT_PATH)


---
### 다시 스왑하고 싶을 때
다른 사진으로 다시 하고 싶으면 셀 **4, 5, 6**만 다시 실행하세요. 셀 1~3은 런타임이 유지되는 동안 다시 실행할 필요 없습니다.

### 결과가 흐릿한가요?
inswapper 모델은 128×128 얼굴 패치를 생성해서 원본이 고해상도면 얼굴 부분이 살짝 부드럽게 보일 수 있어요. 해상도 강화가 필요하면 GFPGAN 같은 후처리 모델을 붙일 수 있는데, 필요하시면 셀 추가해드릴게요.
